# Annotate Treatment Patterns

Classifies each patient into 20 mutually-independent boolean treatment pattern flags
based on their treatment timeline relative to the diagnosis date.
A patient can match multiple rules simultaneously; rule 20 (`other`) fires when none
of rules 1–19 match.

| # | Output column | Pattern |
|---|---|---|
| 1 | `{prefix}only_surgery` | Surgery only |
| 2 | `{prefix}only_radio` | Radiotherapy only |
| 3 | `{prefix}only_chemo` | Chemotherapy only |
| 4 | `{prefix}only_immuno` | Immunotherapy only |
| 5 | `{prefix}only_target` | Targeted therapy only |
| 6 | `{prefix}concomitant_systemic_radio` | Concomitant systemic + radiotherapy |
| 7 | `{prefix}surgery_postop_radio` | Surgery + post-operative radiotherapy |
| 8 | `{prefix}surgery_adj_chemo` | Surgery + adjuvant chemotherapy |
| 9 | `{prefix}surgery_postop_radio_concomi_chemo` | Surgery + post-op radio + concomitant chemo |
| 10 | `{prefix}radio_adj_chemo` | Radiotherapy + adjuvant chemotherapy |
| 11 | `{prefix}concomi_chemo_radio_adj_chemo` | Concomitant chemo-radio + adjuvant chemo |
| 12 | `{prefix}chemo_immuno` | Chemo + immunotherapy |
| 13 | `{prefix}chemo_target` | Chemo + targeted therapy |
| 14 | `{prefix}immuno_target` | Immuno + targeted therapy |
| 15 | `{prefix}neoadj_chemo_radio` | Neoadjuvant chemo → radiotherapy |
| 16 | `{prefix}neoadj_chemo_surgery` | Neoadjuvant chemo → surgery |
| 17 | `{prefix}neoadj_chemo_concomi_chemo_radio` | Neoadj chemo → concomitant chemo-radio |
| 18 | `{prefix}neoadj_chemo_concomi_chemo_radio_adj_chemo` | Neoadj chemo → concomitant chemo-radio → adj chemo |
| 19 | `{prefix}neoadj_chemo_radio_adj_chemo` | Neoadj chemo → radio → adjuvant chemo |
| 20 | `{prefix}other` | None of the above |


In [ ]:
import base64
import json
import requests

In [ ]:
with open("token.txt", "r") as f:
    token = f.read().strip()
headers = {"Authorization": token}

In [ ]:
# Set the collaboration ID:
#
#   - 2: Test collaboration (with IKNL and UPM)
#   - 3: IDEA4RC collaboration
#
COLLABORATION_ID = 3

In [ ]:
# Set the organization IDs that are part of this workspace. These should be the IDs of
# the vantage6 organizations:
#
#   1   - root
#   2   - ENG
#   3   - UPM
#   4   - INT
#   5   - UKE
#   6   - CLB
#   7   - FPNS
#   8   - VGR
#   9   - OUS
#   10  - MSCI
#   11  - APHP
#
# These are basically all organizations that are part of the workspace (thus the same
# list as in the `1-new-workspace.ipynb` notebook):
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/organization?collaboration_id={COLLABORATION_ID}",
    headers=headers
)
ORGANIZATION_IDS = [org["id"] for org in response.json()["data"]]
ORGANIZATION_IDS = [1, 4]

In [ ]:
# Set the study ID. This is the `study` id that belongs to the RAVEN workspace. See the
# `0-new-workspace.ipynb` notebook for more information.
STUDY_ID = 467
# Set the session ID. This is the `v6_session` id that belongs to the RAVEN analysis.
# See the `1-new-analysis.ipynb` notebook for more information.
SESSION_ID = 435

In [ ]:
# In the idea4rc case, preprocessing should always be applied to all cohorts (= vantage6
# dataframes). This because we always want all datasets to have the same columns and
# types.
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/session/{SESSION_ID}/dataframe?per_page=999",
    headers=headers
)
DATAFRAME_IDS = [df["id"] for df in response.json()["data"]]
DATAFRAME_IDS

In [ ]:
# The image to use is the latest version of the preprocessing algorithm.
IMAGE = "ghcr.io/iknl/preprocessing:dev"
#
# The method (within this IMAGE) to execute.
METHOD = "annotate_treatment_patterns"

## REPEAT FOR EACH DATAFRAME

In [ ]:
def payload(df_id):
    return {
        "dataframe_id": df_id,
        "task": {
            "image": IMAGE,
            "method": METHOD,
            "organizations": [
                {
                    "id": org_id,
                    "arguments": base64.b64encode(
                        json.dumps(
                            {
                                # ── Output ──────────────────────────────────────────────────
                                # Prefix for the 20 new boolean columns.
                                # e.g. "trt_pattern_" → trt_pattern_only_surgery, etc.
                                "prefix": "trt_pattern_",

                                # ── General threshold ────────────────────────────────────────
                                # Max days from diagnosis to first treatment for single-modality
                                # rules (1–5) and as a shared upper bound in most other rules.
                                "general_rule_days": 90,

                                # ── Rule 6: concomitant systemic + radio ─────────────────────
                                # Max |systemic_start − radio_start| in days.
                                "concomitant_start_gap": 14,
                                # Max |systemic_end − radio_end| in days.
                                "concomitant_end_gap": 14,

                                # ── Rules 7 & 9: surgery + post-op radio ────────────────────
                                # Max days from surgery to radio start.
                                "surgery_postop_radio_days": 120,

                                # ── Rule 8: surgery + adjuvant chemo ────────────────────────
                                # Max days from surgery to chemo start.
                                "surgery_adjuvant_chemo_days": 120,

                                # ── Rule 9: surgery + post-op radio + concomitant chemo ──────
                                # Max |chemo_start − radio_start| in days.
                                "postop_radio_concomi_start_gap": 14,
                                # Max |chemo_end − radio_end| in days.
                                "postop_radio_concomi_end_gap": 14,

                                # ── Rule 10: radio + adjuvant chemo ─────────────────────────
                                # Max days from radio end to chemo start.
                                "radio_adjuvant_chemo_days": 120,

                                # ── Rule 11: concomitant chemo-radio + adjuvant chemo ────────
                                # Max |chemo1_start − radio_start| in days.
                                "concomi_radio_adj_start_gap": 14,
                                # Max |chemo1_end − radio_end| in days.
                                "concomi_radio_adj_end_gap": 14,
                                # Max days from max(chemo1_end, radio_end) to chemo2_start.
                                "concomi_radio_adj_to_next": 90,

                                # ── Rule 12: chemo + immuno ──────────────────────────────────
                                # Max days from diagnosis to chemo/immuno start.
                                "chemo_immuno_days": 180,

                                # ── Rule 15: neoadj chemo → radio ───────────────────────────
                                # Max days from chemo end to radio start.
                                "neoadj_chemo_to_radio": 90,

                                # ── Rule 16: neoadj chemo → surgery ─────────────────────────
                                # Max days from chemo end to surgery.
                                "neoadj_chemo_to_surgery": 90,

                                # ── Rules 17 & 18: neoadj chemo → concomitant chemo-radio ────
                                # Max days from chemo1 end to chemo2/radio start.
                                "neoadj_concomi_to_phase": 90,
                                # Max |chemo2_start − radio_start| in days.
                                "neoadj_concomi_chemo2_start_gap": 14,
                                # Max |chemo2_end − radio_end| in days.
                                "neoadj_concomi_chemo2_end_gap": 14,

                                # ── Rule 18: + adjuvant chemo ────────────────────────────────
                                # Max days from max(chemo2_end, radio_end) to chemo3_start.
                                "neoadj_concomi_adj_to_next": 90,

                                # ── Rule 19: neoadj chemo → radio → adjuvant chemo ───────────
                                # Max days from chemo1 end to radio start.
                                "neoadj_radio_adj_chemo1_to_radio": 90,
                                # Max days from radio end to chemo2 start.
                                "neoadj_radio_adj_chemo2_to_chemo": 90,
                            }
                        ).encode("UTF-8")
                    ).decode("UTF-8")
                }
                for org_id in ORGANIZATION_IDS  # repeat for each org
            ],
        }
    }

In [ ]:
for df_id in DATAFRAME_IDS:
    response = requests.post(
        f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/session/dataframe/{df_id}/preprocess",
        headers=headers,
        json=payload(df_id)
    )
    print(response.json())
# In the response we need to extract the task ID and the job ID so we can poll whether
# the (central) task is finished. Later on we can also use these IDs to retrieve the
# results.